# Quick start: A Spray of Hadrons

This notebook is a first look at the jet toy of the course and its data. It generates a small data set of QCD, W and
top jets, saves it and reads it back, and shows what is stored: the JetClass branches and the truth record, one jet
at every level of the ladder, the observables, the Lund plane, the network input and a first tagger.

Run it in the environment of the [README](../README.md) (`python -m pip install -e ".[analysis]"`), started with
`python -m jupyter lab` from the repository root, on the kernel `Python (inside-the-jet)`. It takes about a minute on a
CPU and writes one small file to the repository's `data/` folder, which is `../data/` from this notebook's folder. The
physics is in [docs/physics.md](../docs/physics.md), the observables in [docs/observables.md](../docs/observables.md),
the file layout in [docs/data_format.md](../docs/data_format.md) and the tasks in [TASKS.md](../TASKS.md).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

import itj
from itj import plot
from itj.data import Ragged
from itj.features import JetDataset, inputs, make_batch, split
from itj.metrics import auc, one_vs_one, rejection, roc
from itj.models import build_model
from itj.observables import compute, lund_density, lund_primary
from itj.toy import PRESETS, check_identities, generate, load, matching_preset, preset, save, show_ladder
from itj.train import fit, predict

print("itj", itj.__version__, "| torch", torch.__version__)

## The presets

A configuration (`SimConfig`) has six parts: the hard process, the shower, the hadronisation, the underlying event,
the detector and the jet definition. The four presets form a cumulative ladder, each switching on more physics; the
`detector` preset is the default and the level that resembles JetClass. Every parameter can be changed on top.

In [ ]:
switches = ("shower", "hadronize", "b_hadrons", "ue", "detector")
for name, cfg in PRESETS.items():
    flat = {key: value for part in cfg.to_dict().values() for key, value in part.items()}
    print(f"{name:<9s}", "  ".join(f"{s}={'on' if flat[s] else 'off':<3s}" for s in switches))

print()
print(PRESETS["detector"].describe())

In [ ]:
custom = preset("detector", coherence=False)  # the detector preset without colour coherence in the shower
print("coherence:", custom.shower.coherence, "| matching preset:", matching_preset(custom))

## Generate

`generate` draws the hard processes (QCD jets, W -> qq' and t -> bqq' in equal numbers), showers and hadronises them,
adds the underlying event, simulates the detector and clusters the jets. The same seed gives the same hard events at
every preset, so a change after the hard process can be studied jet by jet.

In [ ]:
data = generate(3000, seed=1)
data

The provenance records how the data were made. The containment efficiency is the fraction of hard events whose decay
partons all fall inside one jet: the toy keeps only those, as JetClass does with its truth matching.

In [ ]:
keys = ("seed", "preset", "processes", "containment_efficiency", "package_version", "wall_time_s", "stage_times_s")
{key: data.provenance.get(key) for key in keys}

## What a data set holds

A `JetData` is a table with one row per jet. The JetClass branches (`part_*`, `label_*`, `jet_*`, `aux_*`) are what a
tagger may see; the `truth_*` columns are the toy's truth record (prongs, partons, every shower branching, the primary
Lund plane) and are never inputs to a tagger.

In [ ]:
jetclass = [name for name in data.names if not name.startswith("truth_")]
truth = [name for name in data.names if name.startswith("truth_")]
print(f"{len(jetclass)} JetClass branches, {len(truth)} truth columns")
for name in ("part_px", "part_isPhoton", "label_Wqq", "jet_sdmass", "truth_part_prong", "truth_emis_z"):
    column = data[name]
    kind = "ragged" if isinstance(column, Ragged) else "per jet"
    print(f"    {name:<18s} {column.dtype!s:<8s} {kind}")

A per-particle column is a `Ragged` array: the values of all jets in one flat array and the row boundaries in
`offsets`, so the particles of jet `i` are `values[offsets[i]:offsets[i + 1]]`, sorted by decreasing $p_T$.

In [ ]:
px = data["part_px"]
print("offsets:", px.offsets[:4], "| counts:", px.counts()[:3], "| jet_nparticles:", data["jet_nparticles"][:3])
print("first jet, first particles:", px[0][:4])
print("check identities:", all(check_identities(data).values()))

## Save and load

A file is Parquet with the JetClass schema (the same branches, types and order as the JetClass files) plus the truth
columns, and it keeps the configuration and the provenance.

In [ ]:
Path("../data").mkdir(exist_ok=True)
path = save(data, "../data/quickstart_notebook.parquet", overwrite=True)
again = load(path)
print(again)
print("same configuration:", again.config == data.config)
same = all(
    np.array_equal(a.values, b.values, equal_nan=True) and np.array_equal(a.offsets, b.offsets)
    if isinstance(a, Ragged)
    else np.array_equal(a, b, equal_nan=True)
    for a, b in ((data[name], again[name]) for name in data.names)
)
print("same columns:", same)  # NaN marks a quantity a jet does not have, e.g. cos theta* of a QCD jet
again.to_pandas()[["jet_pt", "jet_sdmass"]].describe().round(1)

## One jet up the ladder

`show_ladder` generates one jet at the four presets: the decay partons, the parton shower, the hadrons with the
underlying event and the detector particles. The colour is the prong (decay quark) each particle comes from; grey is
the underlying event.

In [ ]:
jets = show_ladder("tbqq", seed=1)
fig = plot.ladder(jets)
plt.show()
print({name: int(jet["jet_nparticles"][0]) for name, jet in jets.items()})

In [ ]:
prong = jets["detector"]["truth_part_prong"][0]
pt = np.hypot(jets["detector"]["part_px"][0], jets["detector"]["part_py"][0])
for k in np.unique(prong):
    name = "underlying event" if k < 0 else f"prong {k}"
    print(f"{name:<17s} {np.sum(prong == k):3d} particles, {pt[prong == k].sum() / pt.sum():6.1%} of the pT")

## Observables

The files hold the observables of JetClass: the soft-drop mass $m_\mathrm{SD}$ and the N-subjettiness $\tau_N$.
Their ratios $\tau_{21} = \tau_2 / \tau_1$ and $\tau_{32} = \tau_3 / \tau_2$ count prongs. `itj.observables`
recomputes them from the particles (and many more: `compute` lists them).

In [ ]:
frame = data.to_pandas()[["jet_sdmass", "jet_tau1", "jet_tau2", "jet_tau3"]]
frame["process"] = data.process_names
frame["tau21"] = frame.jet_tau2 / frame.jet_tau1
frame["tau32"] = frame.jet_tau3 / frame.jet_tau2
frame.groupby("process")[["jet_sdmass", "tau21", "tau32"]].median().round(3)

In [ ]:
obs = compute(data, ["sdmass", "tau21", "tau32", "d2"])
stored = np.asarray(data["jet_sdmass"], dtype=np.float64)
print("m_SD recomputed: max |difference| =", np.abs(obs["sdmass"] - stored).max().round(4), "GeV")

fig, ax = plt.subplots(figsize=(6, 4))
for process in ("qcd", "wqq", "tbqq"):
    sel = data.process_names == process
    ax.hist(
        obs["sdmass"][sel], bins=60, range=(0, 300), histtype="step", lw=1.5, color=plot.palette[process], label=process
    )
ax.set_xlabel(r"$m_\mathrm{SD}$ [GeV]")
ax.set_ylabel("jets per 5 GeV")
ax.legend()
plt.show()

## The Lund plane of one jet

Declustering a jet with the Cambridge/Aachen algorithm and following the harder branch gives one point
$(\ln 1/\Delta, \ln k_t)$ per primary emission (`lund_primary`). The toy also stores the primary emissions of its
shower (`truth_lund_*`): the reconstructed points follow them, smeared by hadronisation, the underlying event and the
detector.

In [ ]:
i = int(np.flatnonzero(data.process_names == "qcd")[0])
one = data.select([i])
lund = lund_primary(*one.padded_p4())

fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.scatter(one["truth_lund_ln_inv_delta"][0], one["truth_lund_ln_kt"][0], marker="x", color="k", label="shower (truth)")
ax.scatter(lund["ln_inv_delta"][0], lund["ln_kt"][0], s=18, color=plot.palette["qcd"], label="detector particles")
ax.set_xlabel(r"$\ln(1/\Delta)$")
ax.set_ylabel(r"$\ln(k_t / \mathrm{GeV})$")
ax.set_title(f"jet {i}, {len(lund['z'][0])} primary declusterings")
ax.legend()
plt.show()

In [ ]:
qcd = data.select(np.flatnonzero(data.process_names == "qcd"))
x_edges, y_edges = np.linspace(0, 6, 25), np.linspace(-3, 6, 37)
density, _ = lund_density(lund_primary(*qcd.padded_p4()), n_jets=len(qcd), x_edges=x_edges, y_edges=y_edges)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
plot.lund_plane(density, x_edges, y_edges, ax=ax, n_jets=len(qcd))
plt.show()

## The network input

`inputs` computes the particle features exactly as weaver does from the data configs of the course (`kin`: 7
features per particle); `make_batch` turns jets into a `JetBatch` of torch tensors, padded to the longest jet with a
mask.

In [ ]:
x = inputs(data, "kin")
print({name: array.shape for name, array in x.items()})
batch = make_batch(data.head(8), "kin")
print("features", tuple(batch.features.shape), "| mask", tuple(batch.mask.shape), "| labels", batch.label.tolist())
print(
    "particles per jet:",
    batch.mask.sum(dim=(1, 2)).int().tolist(),
    "=",
    data["jet_nparticles"][:8].astype(int).tolist(),
)

## A first tagger

A cut-based W tagger in four lines: a window in $m_\mathrm{SD}$ and $1 - \tau_{21}$ as the score, W jets against
QCD jets. Then a Particle Flow Network trained for five epochs (seconds) on the same jets, split by the rule of the course (test
= the first 20 %, val = the next 10 %, train = the rest).

In [ ]:
names = data.process_names
w_or_qcd = (names == "wqq") | (names == "qcd")
score = np.where((obs["sdmass"] > 65) & (obs["sdmass"] < 105), 1 - obs["tau21"], -1.0)
print(f"cuts, W vs QCD: AUC {auc(names[w_or_qcd] == 'wqq', score[w_or_qcd]):.3f}")

In [ ]:
train, val, test = split(data)
model = build_model("pfn")
history = fit(model, JetDataset(train), JetDataset(val), epochs=5, seed=0)
probs = predict(model, test)
y = test.labels()

curves = {}
for name, signal in (("W vs QCD", 1), ("top vs QCD", 2)):
    ys, ss, _ = one_vs_one(probs, y, signal, 0)
    eff_s, eff_b, _ = roc(ys, ss)
    curves[name] = (eff_s, eff_b)
    print(f"PFN, {name}: AUC {auc(ys, ss):.3f}, R50 {rejection(ys, ss, 0.5):.1f} ({len(test)} test jets)")
plot.roc_curves(curves)
plt.show()

## Next steps

- The tasks are in [TASKS.md](../TASKS.md): T0 to T6, and the bonus T7.
- `python examples/make_task_data.py --week 1` makes the data sets of the first week (T1 to T3), `--week 2` those of
  the second; on a cluster, see [examples/cluster](../examples/cluster/README.md).
- [01_jetclass.ipynb](01_jetclass.ipynb) does the same on JetClass and with the pretrained ParT.